# BrainInsight AI
# Notebook 09: SHAP & LIME (Stable Version)

This notebook explains the Random Forest model using SHAP and LIME.
It generates:
- SHAP Summary Plot
- SHAP Bar Plot
- LIME HTML Report
- LIME Feature Importance Plot


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import joblib
import shap
import lime.lime_tabular
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..")

OUTPUT_DIR = PROJECT_ROOT / "outputs"
SHAP_DIR = OUTPUT_DIR / "shap"
LIME_DIR = OUTPUT_DIR / "lime"

SHAP_DIR.mkdir(parents=True, exist_ok=True)
LIME_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = PROJECT_ROOT / "saved_models"

model = joblib.load(MODEL_DIR / "random_forest.pkl")
encoder = joblib.load(MODEL_DIR / "label_encoder.pkl")

X = np.load(OUTPUT_DIR / "X.npy")
y = np.load(OUTPUT_DIR / "y.npy", allow_pickle=True)

print("Dataset Shape:", X.shape)


Dataset Shape: (7200, 34658)


In [2]:
# ==========================
# SHAP
# ==========================

explainer = shap.TreeExplainer(model)

sample_size = min(100, len(X))
X_sample = X[:sample_size]

shap_values = explainer.shap_values(X_sample)

plt.figure()
shap.summary_plot(shap_values, X_sample, show=False)
plt.tight_layout()
plt.savefig(SHAP_DIR / "summary_plot.png", dpi=300)
plt.close()

plt.figure()
shap.summary_plot(
    shap_values,
    X_sample,
    plot_type="bar",
    show=False
)
plt.tight_layout()
plt.savefig(SHAP_DIR / "bar_plot.png", dpi=300)
plt.close()

print("SHAP plots generated successfully.")


SHAP plots generated successfully.


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [3]:
# ==========================
# LIME
# ==========================

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X,
    feature_names=[f"Feature_{i}" for i in range(X.shape[1])],
    class_names=list(encoder.classes_),
    mode="classification"
)

instance = X[0]

explanation = lime_explainer.explain_instance(
    instance,
    model.predict_proba,
    num_features=15
)

explanation.save_to_file(str(LIME_DIR / "lime_report.html"))

fig = explanation.as_pyplot_figure()
fig.savefig(
    LIME_DIR / "lime_feature_weights.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)

print("LIME outputs generated successfully.")


LIME outputs generated successfully.


In [4]:
print("\nGenerated Files")

print("\nSHAP")
for file in sorted(SHAP_DIR.iterdir()):
    print("-", file.name)

print("\nLIME")
for file in sorted(LIME_DIR.iterdir()):
    print("-", file.name)

print("\nNotebook completed successfully.")



Generated Files

SHAP
- bar_plot.png
- summary_plot.png

LIME
- lime_feature_weights.png
- lime_report.html

Notebook completed successfully.


## Conclusion

This notebook provides stable and fast explainability using SHAP and LIME and is compatible with the project pipeline.
